# CAT Saathi — train the intent model

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

No Google Drive needed. Training runs on Colab's own disk and backs up to a **private**
Hugging Face repo every few minutes. If Colab disconnects, just **Run all** again —
it pulls the backup and carries on. The finished model is published to the same repo.


## 1. Your Hugging Face token
Needs **write** access: huggingface.co/settings/tokens → New token → *Write*. It is typed into a hidden box.


In [ ]:
import os, getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face WRITE token: ')
REPO = 'cat-saathi-intent'   # created privately under your account
print('token set, repo:', REPO)


## 2. GPU check and code


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU - switch runtime to T4'
!git clone -q https://github.com/krishagarwal314/sahayak-cat-operator-assistant.git /content/saathi 2>/dev/null || git -C /content/saathi pull -q
%cd /content/saathi/backend


## 3. Build the training data


In [ ]:
!python -m app.ai.intent.build_dataset


## 4. Train
One progress bar for the whole run, about 5–10 minutes on a T4.
If it stops for any reason, run this cell again — it resumes.


In [ ]:
!python -m app.ai.intent.train --out /content/intent-run --hub-repo $REPO --epochs 6


## 5. Try it
Type anything — English, Hindi or Hinglish.


In [ ]:
import sys, json, torch
sys.path.insert(0, '/content/saathi/backend')
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from app.ai.intent.normalize import normalize
best = '/content/intent-run/best'
tok = AutoTokenizer.from_pretrained(best)
model = AutoModelForSequenceClassification.from_pretrained(best).eval()
labels = {int(k): v for k, v in json.load(open(best + '/labels.json')).items()}

def ask(text):
    with torch.no_grad():
        probs = torch.softmax(model(**tok(normalize(text), return_tensors='pt')).logits[0], -1)
    top = probs.topk(2)
    print(f'{text:42s} -> {labels[top.indices[0].item()]:20s} {top.values[0]:.0%}   (2nd {labels[top.indices[1].item()]} {top.values[1]:.0%})')

for q in ['how much fuel is left', 'is anything wrong with the machine', 'is it safe to work',
          'how long will this take', 'how do i start the machine', 'kitna diesel bacha hai',
          'मशीन में कोई खराबी है क्या', 'how long did i idle', 'how do i make tea']:
    ask(q)


## 6. Put it on Lightning
The last lines of the training output print the exact command. It is this, run from the repo root on Lightning:

```bash
HF_TOKEN=<your token> python -c "from huggingface_hub import snapshot_download; snapshot_download('<your-username>/cat-saathi-intent', local_dir='backend/models/intent-classifier', ignore_patterns=['checkpoint/*'])"
CLASSIFIER_FIRST=1 bash scripts/demo.sh
```
